In [1]:
import pandas as pd
import numpy as np
import joblib
import os

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load Trained Model
MODEL_PATH = os.path.join("models", "best_model.pkl")
MODEL_PATH = os.path.abspath(MODEL_PATH)
# Check if the model file exists
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

model = joblib.load(MODEL_PATH)

print("Best model loaded successfully.")

Best model loaded successfully.


In [3]:
# To view the model parameters
print("Model Parameters:")
for param, value in model.get_params().items():
    print(f"  {param}: {value}")

Model Parameters:
  memory: None
  steps: [('preprocessor', ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Annual Income',
                                  'Number of Dependents', 'Health Score',
                                  'Previous Claims', 'Vehicle Age',
                                  'Credit Score', 'Insurance Duration',
                                  'Policy_Year', 'Policy_Month',
                                  'Policy_Day']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                 

In [4]:
# Load Test Dataset
TEST_DATA_PATH = os.path.join("data", "playground-series-s4e12", "test.csv")
TEST_DATA_PATH = os.path.abspath(TEST_DATA_PATH)

# Check if the test data file exists
if not os.path.exists(TEST_DATA_PATH):
    raise FileNotFoundError(f"Test data not found: {TEST_DATA_PATH}")   

test_df = pd.read_csv(TEST_DATA_PATH)
print("Test Shape :", test_df.shape)

# Display the first few rows of the test dataset
test_df.head()

Test Shape : (800000, 20)


,id,Age,Gender,Annual Income,Marital Status,Number of Dependents,Education Level,Occupation,Health Score,Location,Policy Type,Previous Claims,Vehicle Age,Credit Score,Insurance Duration,Policy Start Date,Customer Feedback,Smoking Status,Exercise Frequency,Property Type
0,1200000,28.0,Female,2310.0,NaN,4.0,Bachelor's,Self-Employed,7.657981,Rural,Basic,NaN,19.0,NaN,1.0,2023-06-04 15:21:39.245086,Poor,Yes,Weekly,House
1,1200001,31.0,Female,126031.0,Married,2.0,Master's,Self-Employed,13.381379,Suburban,Premium,NaN,14.0,372.0,8.0,2024-04-22 15:21:39.224915,Good,Yes,Rarely,Apartment
2,1200002,47.0,Female,17092.0,Divorced,0.0,PhD,Unemployed,24.354527,Urban,Comprehensive,NaN,16.0,819.0,9.0,2023-04-05 15:21:39.134960,Average,Yes,Monthly,Condo
3,1200003,28.0,Female,30424.0,Divorced,3.0,PhD,Self-Employed,5.136225,Suburban,Comprehensive,1.0,3.0,770.0,5.0,2023-10-25 15:21:39.134960,Poor,Yes,Daily,House
4,1200004,24.0,Male,10863.0,Divorced,2.0,High School,Unemployed,11.844155,Suburban,Premium,NaN,14.0,755.0,7.0,2021-11-26 15:21:39.259788,Average,No,Weekly,House


In [5]:
submission_ids = test_df["id"]
submission_ids.head()

0    1200000
1    1200001
2    1200002
3    1200003
4    1200004
Name: id, dtype: int64

In [6]:
# Drop unwanted columns
test_df.drop(["id", "Customer Feedback"], axis=1, inplace=True)

# Convert to datetime
test_df["Policy Start Date"] = pd.to_datetime(test_df["Policy Start Date"], errors="coerce")

# Extract date features
test_df["Policy_Year"] = test_df["Policy Start Date"].dt.year
test_df["Policy_Month"] = test_df["Policy Start Date"].dt.month
test_df["Policy_Day"] = test_df["Policy Start Date"].dt.day

# Remove original date column
test_df.drop("Policy Start Date", axis=1, inplace=True)

# Check result
print("After feature engineering:", test_df.shape)
test_df.head()

After feature engineering: (800000, 20)


,Age,Gender,Annual Income,Marital Status,Number of Dependents,Education Level,Occupation,Health Score,Location,Policy Type,Previous Claims,Vehicle Age,Credit Score,Insurance Duration,Smoking Status,Exercise Frequency,Property Type,Policy_Year,Policy_Month,Policy_Day
0,28.0,Female,2310.0,NaN,4.0,Bachelor's,Self-Employed,7.657981,Rural,Basic,NaN,19.0,NaN,1.0,Yes,Weekly,House,2023,6,4
1,31.0,Female,126031.0,Married,2.0,Master's,Self-Employed,13.381379,Suburban,Premium,NaN,14.0,372.0,8.0,Yes,Rarely,Apartment,2024,4,22
2,47.0,Female,17092.0,Divorced,0.0,PhD,Unemployed,24.354527,Urban,Comprehensive,NaN,16.0,819.0,9.0,Yes,Monthly,Condo,2023,4,5
3,28.0,Female,30424.0,Divorced,3.0,PhD,Self-Employed,5.136225,Suburban,Comprehensive,1.0,3.0,770.0,5.0,Yes,Daily,House,2023,10,25
4,24.0,Male,10863.0,Divorced,2.0,High School,Unemployed,11.844155,Suburban,Premium,NaN,14.0,755.0,7.0,No,Weekly,House,2021,11,26


In [7]:
predictions = model.predict(test_df)

print("Prediction Completed")
print(predictions[:5])

Prediction Completed
[1462.86205393 1121.77446394 1119.46803629 1096.28684747 1040.32741575]


In [8]:
submission = pd.DataFrame({
    "id": submission_ids,
    "Premium Amount": predictions
})

submission.head()

,id,Premium Amount
0,1200000,1462.862054
1,1200001,1121.774464
2,1200002,1119.468036
3,1200003,1096.286847
4,1200004,1040.327416


In [9]:
OUTPUT_PATH = os.path.join("data", "submission.csv")

submission.to_csv(OUTPUT_PATH, index=False)

print(f"Submission file saved at {OUTPUT_PATH}")

Submission file saved at data\submission.csv


In [10]:
submission.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800000 entries, 0 to 799999
Data columns (total 2 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              800000 non-null  int64  
 1   Premium Amount  800000 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 12.2 MB


In [11]:
submission.head()

,id,Premium Amount
0,1200000,1462.862054
1,1200001,1121.774464
2,1200002,1119.468036
3,1200003,1096.286847
4,1200004,1040.327416
